# Classical RAG: The Canonical Pipeline

**Difficulty:** Beginner | **Time:** ~40 min | **Requires:** None — this is the first lab

---

## The problem

A restoration team has a guide on wetland restoration and needs to answer questions about it without reading it by hand. A plain LLM cannot help — it has never seen this document, so it makes things up. You will build a pipeline that finds the right paragraphs first, then answers using only those paragraphs.

## What RAG actually is

Suppose you need to answer a question from a document. The reliable way is not to recall it from memory — it is to **open the document, find the relevant passage, and answer from that passage.**

That is RAG:

1. You get a question.
2. You **search the document** and **find the passage** that answers it.
3. You **hand that passage to an LLM**, which writes the answer using only that text.

An LLM on its own does something different. It answers from whatever it memorised during training, and it has **never seen your document**, your handbook, or your notes. So when you ask it about your document it does one of two things:

- it says **"I don't know"**, or
- it **makes something up** that sounds right.

That second failure mode is called a **hallucination**, and it is the biggest practical problem with plain LLMs.

**RAG** = **R**etrieval-**A**ugmented **G**eneration. In plain words:

> **Find the right passage in your documents first, then answer using only that passage.**

That is the whole idea. Everything else in this lab is the plumbing that makes those three words happen.

```mermaid
flowchart LR
    Q["Your question<br/>'Why is the sky blue?'"] --> R["RETRIEVAL<br/>Search my document<br/>find the best-matching paragraphs"]
    D["My document<br/>(the model has never seen it)"] --> R
    R --> A["AUGMENTATION<br/>Stick those paragraphs<br/>onto the question"]
    A --> G["GENERATION<br/>The LLM answers<br/>using ONLY the paragraphs"]
    G --> O["Grounded answer<br/>+ the sources it used"]

    classDef input fill:#e1f5ff,stroke:#0288d1,stroke-width:2px,color:#0d3b66
    classDef mid fill:#fff9c4,stroke:#f9a825,stroke-width:2px,color:#5c4300
    classDef out fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px,color:#1b4d1f
    class Q,D input
    class R,A,G mid
    class O out
```

Read that as a story: the question and the document both flow into **Retrieval**, retrieval feeds **Augmentation**, augmentation feeds **Generation**, and out comes a grounded answer.

**Why it matters in real life**

- **Private documents** — contracts, manuals, papers, internal wikis, your own notes. The model was never trained on them.
- **Answers you can check** — this lab prints the exact paragraphs behind every answer.
- **No retraining** — drop in a new document and it works.

## The Six Steps We'll Build

1. **Load** the document → 2. **Chunk** the text → 3. **Embed** each chunk into numbers → 4. **Index** them for search → 5. **Retrieve** the closest ones → 6. **Generate** a grounded answer.

```mermaid
flowchart LR
    D["Text document"] --> L["1. LOAD<br/>read_text()"]
    L --> C["2. CHUNK<br/>400 chars, 50 overlap"]
    C --> E["3. EMBED<br/>all-MiniLM-L6-v2<br/>384 unit numbers each"]
    E --> V[("4. INDEX<br/>FAISS IndexFlatIP<br/>inner product = cosine")]
    Q["Your question"] --> R["5. RETRIEVE<br/>top-k closest chunks"]
    V --> R
    R --> G["6. GENERATE<br/>LLM, context only"]
    G --> A["Grounded answer<br/>+ the sources used"]

    classDef input fill:#e1f5ff,stroke:#0288d1,stroke-width:2px,color:#0d3b66
    classDef proc fill:#fff9c4,stroke:#f9a825,stroke-width:2px,color:#5c4300
    classDef out fill:#c8e6c9,stroke:#2e7d32,stroke-width:2px,color:#1b4d1f
    class D,Q input
    class L,C,E,V,R,G proc
    class A out
```

Follow the arrows: the document takes the left path down to the index, the question takes the short path to retrieval, they meet at retrieval, and the result flows out to the right.

This is deliberately the **baseline**. Every later lab in this module changes exactly one of these six stages — a smarter chunker, a second retriever, a graph instead of a flat index, or a loop that double-checks the retriever. You cannot judge those upgrades until you have felt this one's limits yourself.

---

## Setup

### Install Dependencies

One command installs everything. `-q` means quiet, `-U` upgrades an older install if present.

In [1]:
!pip install -qU langchain langchain-community langchain-openai langchain-huggingface sentence-transformers langchain-text-splitters faiss-cpu python-dotenv

| Package | Job it does for you |
|---------|---------------------|
| `langchain` | The `Document` type and the chain-building pieces |
| `langchain-community` | `FAISS`, the vector store that holds and searches the embeddings |
| `langchain-huggingface` | Wraps the `all-MiniLM-L6-v2` embedding model in a LangChain interface |
| `sentence-transformers` | **The engine that actually runs the embedding model.** `langchain-huggingface` treats this as an optional extra and will *not* install it for you, so it is listed explicitly — skip it and Step 3 dies with `ImportError: Could not import sentence_transformers`. It brings PyTorch with it, which is why this install is the slowest step. |
| `langchain-text-splitters` | `RecursiveCharacterTextSplitter`, which does the cutting |
| `langchain-openai` | `ChatOpenAI`, used to talk to OpenRouter |
| `faiss-cpu` | The vector-search engine underneath `FAISS` |
| `python-dotenv` | Loads `OPENROUTER_API_KEY` from a `.env` file |

**You also need an OpenRouter API key** for Step 5 — the only step that uses the internet.

1. Go to [openrouter.ai](https://openrouter.ai) and sign up or log in.
2. Open the **Keys** section of the dashboard.
3. Click **Create Key**, name it, confirm.
4. **Copy it immediately** — it is shown in full only once.
5. Put it in a `.env` file in this same folder:

```
OPENROUTER_API_KEY=sk-or-v1-...
```

The notebook reads that file automatically and falls back to prompting you, so a missing `.env` will not crash the run.

### Silence the Noisy Libraries

These libraries print progress bars and warnings that bury the actual output. This cell touches nothing in the pipeline; it just quiets things down so you can see your results.

**This cell runs *before* the imports on purpose.** Two things happen at import time that we want already silenced: LangChain's own "this package is being deprecated" notice, and the download progress bar for the embedding model. Silence first, then import — otherwise the noise is already on screen.

In [2]:
# os lets us read and set environment variables.
import os

# Tell HuggingFace not to draw download progress bars. Without this you get a
# noisy "Loading weights: 0%|..." bar in the middle of your results.
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Warnings are mostly about model internals we never touch in this lab.
import warnings
import logging

warnings.filterwarnings("ignore")

# These libraries log every model download and every HTTP call at "info"
# level, which drowns out our own print statements.
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("httpx").setLevel(logging.ERROR)

> **Two lines you may still see, both harmless.**
> - On the very first run, HuggingFace prints a note suggesting you set an `HF_TOKEN` for faster downloads. **Ignore it** — downloading anonymously works fine, and it only appears once.
> - `langchain-community` is officially deprecated upstream, but it still works, and every lab in this module uses it. We silence its notice rather than rewriting the module.

### Import the Tools

LangChain splits its helpers into small packages, one per job. The imports below are grouped by that job so you can see where everything comes from.

In [3]:
# --- 1. READING THE DOCUMENT -----------------------------------------
# pathlib is part of Python itself, so there is nothing to install here.
# Path.read_text() is the whole "loader" for a plain-text file.
from pathlib import Path
# Document is the atom every RAG system passes around. A text loader would
# have built these for us; here we build them ourselves, which is worth
# seeing once so you know what is inside.
from langchain_core.documents import Document

# --- 2. CUTTING THE TEXT INTO CHUNKS ----------------------------------
# RecursiveCharacterTextSplitter cuts long text into small, tidy pieces.
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- 3. TURNING TEXT INTO NUMBERS, AND SEARCHING THOSE NUMBERS -------
# HuggingFaceEmbeddings runs the all-MiniLM-L6-v2 model on your own machine.
from langchain_huggingface import HuggingFaceEmbeddings
# FAISS is the vector index: the searchable store of embeddings.
from langchain_community.vectorstores import FAISS
# DistanceStrategy lets us choose HOW similarity is measured. We use
# MAX_INNER_PRODUCT — see the warning in Step 3, it matters more than it looks.
from langchain_community.vectorstores.utils import DistanceStrategy

# --- 4. CALLING THE LLM AND BUILDING THE PROMPT -----------------------
# ChatOpenAI talks to any OpenAI-compatible API, which includes OpenRouter.
from langchain_openai import ChatOpenAI
# ChatPromptTemplate builds the instruction with holes ({context}, {question})
# that we fill in later.
from langchain_core.prompts import ChatPromptTemplate
# StrOutputParser turns the model's reply object into a plain Python string.
from langchain_core.output_parsers import StrOutputParser

# --- 5. KEEPING THE API KEY OUT OF THE CODE ---------------------------
# load_dotenv reads our API key out of the .env file.
from dotenv import load_dotenv

- Every LangChain import is `from <package> import <thing>`, where the package name tells you the job: `document_loaders` reads files, `text_splitters` cuts text, `vectorstores` stores and searches vectors, `prompts` builds prompts.
- Nothing here does any work yet. This cell only makes the tools available.

### Load the API Key and Build the Two Models

This cell authenticates the LLM and loads the embedding model once, so every later step reuses the same objects instead of reloading them.

In [4]:
# Read OPENROUTER_API_KEY from the .env file sitting next to this notebook.
load_dotenv(".env")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

# If there is no .env file, ask for the key instead of crashing.
if not OPENROUTER_API_KEY:
    OPENROUTER_API_KEY = input("Paste your OpenRouter API key: ").strip()

print("API key loaded.")

# THE LLM — this is the "brain" that writes the final answer in Step 5.
# It runs on someone else's computer, so it needs the key.
llm = ChatOpenAI(
    model="nvidia/nemotron-3-ultra-550b-a55b:free",  # a free model on OpenRouter
    base_url="https://openrouter.ai/api/v1",         # OpenRouter's OpenAI-compatible door
    api_key=OPENROUTER_API_KEY,
    temperature=0,  # 0 = as repeatable as the model allows
)

# THE EMBEDDING MODEL — this is the "translator" that turns text into numbers.
# It runs on YOUR computer, so it costs nothing and needs no key. The first
# run downloads ~90 MB and then remembers it.
embedding_model = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",              # small, fast, good enough for this lab
    encode_kwargs={"normalize_embeddings": True},  # ask for unit-length vectors
)

print("LLM ready:", llm.model_name)
print("Embedding model ready: 384 numbers per chunk")

API key loaded.


LLM ready: nvidia/nemotron-3-ultra-550b-a55b:free
Embedding model ready: 384 numbers per chunk


- **There are two models in this lab, and they do completely different jobs.** The embedding model (local, free) converts text to numbers so the computer can *compare* it. The LLM (remote, needs a key) *writes prose*. Steps 1–4, 6 and 7 never touch the LLM, so if your key is wrong you will still get real retrieval results — a useful thing to know when something breaks.
- `normalize_embeddings=True` **asks** for unit-length vectors. This model already returns them, so the setting is belt-and-braces rather than a rescue — but stating it means the next stage does not depend on a default you did not choose.
- `temperature=0` makes the answer repeatable, so re-running gives you a near-identical answer.
- Both objects are created **once** here. Reloading an embedding model takes seconds, and a later step would silently get a *different* model than the index was built with.

---

### Step 1 — Load the Document

This step turns the file into `Document` objects the rest of the pipeline can cut up.

In [5]:
# The path is relative, so Jupyter must be running from inside this lab folder.
DOC_PATH = "data/sample_text_document.txt"

# read_text() reads the whole file and hands back one long Python string.
raw_text = Path(DOC_PATH).read_text(encoding="utf-8")

# "\f" is the form-feed character -- the traditional ASCII page break in
# plain text. Splitting on it gives us one piece per page.
page_texts = [text.strip() for text in raw_text.split("\f") if text.strip()]

# Wrap each page in a Document: the text, plus a metadata dict recording
# where it came from. This metadata survives chunking, which is the point.
pages = [
    Document(page_content=text, metadata={"source": DOC_PATH, "page": number})
    for number, text in enumerate(page_texts)
]

total_characters = sum(len(page.page_content) for page in pages)

print(f"Loaded {len(pages)} page(s), {total_characters:,} characters.")

Loaded 2 page(s), 4,062 characters.


- A LangChain **`Document`** is the atom of every RAG system: some text plus a metadata dictionary. Everything downstream — chunking, embedding, retrieval — operates on `Document` objects, never on bare strings.
- **`"\f"` means the form-feed character**, not a backslash and an f. Writing `split("\\f")` would split on the two literal characters instead and give you one single page back — a quiet bug worth knowing.
- **Why strip each page?** The raw string is 4,064 characters while the document is really 4,062 — the two extras are the form-feed character itself and the trailing newline at end of file. `.strip()` on each page is what makes the count honest.
- `enumerate(...)` numbers the pages from `0`, which is why Step 4 prints `page 0` for the first one. That is a computer habit; a human would call it page 1.
- **What we gave up by dropping the PDF:** nothing downstream. A real loader only decides how the string gets out of the file. Every stage after this one is byte-identical whether the input was a PDF, a web page, or a `.txt`.
- The comma in `{total_characters:,}` is a Python format trick that inserts thousands separators, so `4062` prints as `4,062`.
- **If this cell errors with `FileNotFoundError`,** your working directory is wrong. Check it with `import os; print(os.getcwd())`.

---

### Step 2 — Chunk the Text

This step slices the long text into small, overlapping pieces — the units the retriever will actually search.

In [6]:
# Each chunk aims to be ~400 characters long...
CHUNK_SIZE = 400
# ...and each new chunk repeats the last 50 characters of the previous one.
CHUNK_OVERLAP = 50

# Build the splitter with those two settings.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
)

# Apply it to every page we loaded in Step 1.
chunks = text_splitter.split_documents(pages)

print(f"Created {len(chunks)} chunks ({CHUNK_SIZE} chars, {CHUNK_OVERLAP} overlap).")
print("\nFirst chunk:\n" + chunks[0].page_content)

Created 13 chunks (400 chars, 50 overlap).

First chunk:
A Short Guide to Coastal Wetland Restoration
Synthetic Sample Document — Original Content, No Copyright Restrictions
1. Introduction
Coastal wetlands sit at the boundary between land and sea, absorbing storm surge, filtering runoff, and
providing nursery habitat for fish and shellfish. Over the past century, many of these ecosystems have


- **`split_documents` returns `Document` objects, not strings.** That is what lets the page number survive the cut — every chunk still knows where it came from.
- **Why "recursive"?** It tries to cut at paragraph breaks first, then line breaks, then sentences, and only cuts mid-word as a last resort. That is why the first chunk above ends cleanly at a paragraph edge instead of in the middle of the word "have".
- **Why overlap at all?** So an idea that would be cut in half still appears whole in one of the two neighbouring chunks.
- **Common misunderstanding:** `chunk_size=400` counts **characters**, not words and not LLM tokens. 400 characters is roughly 60–70 English words.
- `chunks[0].page_content` is the text; the `.page_content` part is how you get at the text inside a `Document`.

---

### Step 3 — Embed the Chunks and Build the Index

This is where text becomes searchable numbers. LangChain does the embedding *and* the storing in one call.

In [7]:
# from_documents does two things at once:
#   1. runs embedding_model over every chunk  ->  13 lists of 384 numbers
#   2. stores all 13 vectors in a FAISS index that similarity_search can query
vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model,
    # Compare by INNER PRODUCT. Because every vector is unit length, the inner
    # product IS the cosine similarity, so the number comes back readable.
    distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,
)

print("Chunks indexed:", len(chunks))
print("Numbers per vector:", vector_store.index.d)
print("Index type:", type(vector_store.index).__name__)

Chunks indexed: 13
Numbers per vector: 384
Index type: IndexFlatIP


- `vector_store.index.ntotal` would also work for the count; we print `len(chunks)` because it is the same number and does not require knowing FAISS internals.
- `vector_store.index.d` is FAISS's name for the vector width. Every vector in an index must have the same width, which is why the question in Step 4 must be embedded by the same model.
- **One line did two stages.** Doing this by hand was "embed each chunk yourself, then construct a FAISS index yourself, then add the vectors" — three fiddly steps. LangChain wraps them into `from_documents`, which is exactly the kind of shortcut a Beginner lab should take.
- **`MAX_INNER_PRODUCT` builds an `IndexFlatIP` and returns the inner product.** Since our vectors are unit length, that inner product *is* the cosine similarity: `1` identical, `0` unrelated, `-1` opposite, **higher is better**. The output confirms we really got an `IndexFlatIP`.
- **⚠️ This is the lab's biggest stumbling block, so read it twice.** LangChain also offers a strategy literally named `DistanceStrategy.COSINE`, and on this version it quietly builds an **`IndexFlatL2`** and returns **squared straight-line distance** — a different number with the **opposite** direction (lower is better). If you switch to it, your top score comes back as `0.457` where the true cosine is `0.771`, and a *high* score starts meaning *worse*. `0.457` is exactly `2 × (1 − 0.771)`, the squared distance between two unit vectors. **Always confirm which index was built before you trust a score** — that is why this cell prints `Index type:`.

---

### Step 4 — Retrieve the Top-k Chunks

This step embeds your question and asks FAISS which chunks are closest to it.

In [8]:
# The question we want answered. Try changing it later.
QUESTION = "Why do restoration teams reintroduce tidal flow gradually instead of all at once?"

# How many chunks to bring back. 3 is a sensible starting point.
TOP_K = 3

# similarity_search_with_score turns your text into a vector and returns
# (chunk, score) pairs, best first.
matches = vector_store.similarity_search_with_score(QUESTION, k=TOP_K)

print(f"QUESTION: {QUESTION}\n")

for rank, (chunk, score) in enumerate(matches, start=1):
    # With MAX_INNER_PRODUCT the score is already the cosine similarity:
    # higher is better, 1 is identical. No conversion needed.
    print(f"--- Chunk {rank} | page {chunk.metadata['page']} | match {score:.3f} ---")
    print(chunk.page_content.strip() + "\n")

QUESTION: Why do restoration teams reintroduce tidal flow gradually instead of all at once?

--- Chunk 1 | page 0 | match 0.771 ---
3. Reintroducing Tidal Flow
Once barriers are addressed, restoration teams usually reintroduce tidal flow gradually rather than all at
once. A sudden full breach can erode unconsolidated fill material before vegetation has a chance to
stabilize the soil. Phased approaches might open a small channel first, monitor erosion and sediment

--- Chunk 2 | page 0 | match 0.626 ---
tide gates installed decades earlier for agricultural drainage. Removing or resizing these structures is often
the single most effective step in a restoration project, since it allows the natural rise and fall of the tide to
resume moving sediment, nutrients, and organisms in and out of the site.
3. Reintroducing Tidal Flow

--- Chunk 3 | page 0 | match 0.485 ---
been drained, filled, or cut off from tidal flow to make way for farmland, roads, and coastal development.
This guide summariz

- **`matches` is a list of `(chunk, score)` tuples**, best match first. `enumerate(..., start=1)` numbers them 1, 2, 3 so the output reads like "rank 1, rank 2" rather than "index 0, index 1".
- `chunk.metadata['page']` works because the page number was attached by hand in Step 1, then travelled with the text through the splitter in Step 2, into the index in Step 3. **This is the payoff of using `Document` objects instead of plain strings** — you can always say where an answer came from, and the metadata is yours to choose. **This is the payoff of using `Document` objects instead of plain strings** — you can always say where an answer came from.
- `:.3f` formats the number to three decimal places (`0.771`). Without it Python would print `0.7712710331949077`.
- **Now read the three scores, because they are the whole lesson.** Chunk 1 scored `0.771` and it is the section that answers the question. Chunk 2 scored `0.626` — same topic, different point. Chunk 3 scored `0.485`, it is the **introduction**, and it does not answer the question at all.
- **The trap: Chunk 3's score is positive and respectable, and it is still useless.** Notice there is no negative score here and nothing that looks like a rejection. A cosine tells you how similar two texts *sound*; only reading them tells you whether they *answer* what you asked. Nothing in this output marks the good chunk.
- **A retriever never returns "nothing found."** It always hands back `k` chunks. Chunk 3 came back because `k=3` was requested, not because the retriever thought it was useful. Step 7 shows how bad a match actually looks.

---

### Step 5 — Generate the Answer

This step hands the retrieved chunks to the LLM and gets a grounded answer back.

In [9]:
# Glue the retrieved chunks into one block of text for the prompt.
context = "\n\n".join(chunk.page_content.strip() for chunk, _ in matches)

# The prompt template. {context} and {question} are holes we fill in below.
rag_prompt = ChatPromptTemplate.from_template(
    "Answer the question using ONLY the context below. "
    "If the answer is not in the context, say you don't know.\n\n"
    "Context:\n{context}\n\nQuestion: {question}\nAnswer:"
)

# The pipe operator chains three things together, left to right:
#   fill in the prompt  ->  send it to the LLM  ->  keep only the text
rag_chain = rag_prompt | llm | StrOutputParser()

# Run the chain by handing it exactly the names of the holes in the template.
answer = rag_chain.invoke({"context": context, "question": QUESTION})

print(f"Question: {QUESTION}")
print(f"\nAnswer: {answer}")

Question: Why do restoration teams reintroduce tidal flow gradually instead of all at once?

Answer: Restoration teams reintroduce tidal flow gradually because a sudden full breach can erode unconsolidated fill material before vegetation has a chance to stabilize the soil.


- **The `|` symbol is the important bit.** It is LangChain's shorthand for "do this, then hand the result to that". Read `a | b | c` as "a, then b, then c".
- **`invoke` takes a plain dictionary.** The keys must match the hole names in the template exactly — `{"context": ..., "question": ...}`. A typo here raises a `KeyError` naming the variable it wanted, which is a friendly failure.
- **Why the double newline `\n\n`?** It separates chunks with a blank line, which stops the model from reading two unrelated paragraphs as one continuous thought.
- **`StrOutputParser` is what makes `answer` a string.** Without it you would get a full response object with metadata, token counts, and the text buried inside it.
- **This is the only step that needs the API key.** Everything else in the lab is free and offline.
- **If the answer sounds wrong,** do not retrain anything — print `context` and read it. Nine times out of ten the right text was never retrieved, and no amount of prompt tweaking fixes a retrieval miss. Here the answer is nearly a verbatim lift from Chunk 1, which is exactly what "grounded" looks like.

---

### Step 6 — Sweep the Two Settings That Matter

Now that the pipeline runs, this step changes the two knobs that decide how good it is, and prints what each one buys.

In [10]:
# --- Knob 1: TOP_K — how many chunks reach the LLM ---
# No LLM calls here, so this sweep is fast and free.
print("--- Changing TOP_K (how many chunks reach the LLM) ---")
for k in (1, 3, 5):
    trial_matches = vector_store.similarity_search_with_score(QUESTION, k=k)
    context_characters = sum(len(chunk.page_content) for chunk, _ in trial_matches)
    print(f"TOP_K={k}: {k} chunks, {context_characters:,} context chars, best match {trial_matches[0][1]:.3f}")

# --- Knob 2: CHUNK_SIZE — how finely we cut the document ---
print()
print("--- Changing CHUNK_SIZE (how finely we cut the document) ---")
for size in (300, 400, 800):
    # A different chunk size means different chunks, which means we must
    # rebuild the whole index before searching it again.
    trial_splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
    trial_chunks = trial_splitter.split_documents(pages)
    trial_store = FAISS.from_documents(
        documents=trial_chunks,
        embedding=embedding_model,
        distance_strategy=DistanceStrategy.MAX_INNER_PRODUCT,
    )
    best_chunk, best_score = trial_store.similarity_search_with_score(QUESTION, k=1)[0]
    # Show the first few words of whatever won, so we can see *which* section it was.
    opening_words = " ".join(best_chunk.page_content.split()[:8])
    print(f"CHUNK_SIZE={size}: {len(trial_chunks):>2} chunks, best match {best_score:.3f} -> \"{opening_words}...\"")

--- Changing TOP_K (how many chunks reach the LLM) ---
TOP_K=1: 1 chunks, 335 context chars, best match 0.771
TOP_K=3: 3 chunks, 965 context chars, best match 0.771
TOP_K=5: 5 chunks, 1,604 context chars, best match 0.771

--- Changing CHUNK_SIZE (how finely we cut the document) ---
CHUNK_SIZE=300: 18 chunks, best match 0.814 -> "3. Reintroducing Tidal Flow Once barriers are addressed,..."
CHUNK_SIZE=400: 13 chunks, best match 0.771 -> "3. Reintroducing Tidal Flow Once barriers are addressed,..."


CHUNK_SIZE=800:  6 chunks, best match 0.687 -> "3. Reintroducing Tidal Flow Once barriers are addressed,..."


- **Raising `TOP_K` raises recall, not accuracy.** More chunks means the answer is more likely to be somewhere in the context, but also means more irrelevant text competing for the model's attention — and more tokens you pay for. Notice the best match stays `0.771` in every row: asking for more chunks cannot improve your single best chunk, because it was already found at `k=1`.
- **Changing `CHUNK_SIZE` rebuilds the index.** That is why this loop re-splits *and* re-builds instead of just searching again. It is also why chunk size is a more dangerous setting than `TOP_K`: it changes what the index contains, not just how much of it you read.
- `{len(trial_chunks):>2}` right-aligns the count in a 2-wide column so the three rows line up.
- `" ".join(...split()[:8])` collapses all the newlines and spaces, then takes the first 8 words — a tidy one-line preview of the winning chunk.
- **The winning section is the same in all three rows, but the score is not** (0.814 → 0.687). A higher score at `size=300` does not automatically mean a better answer — it can just mean a shorter chunk that happens to sit closer to the question's wording. Score and usefulness are two different things.
- **There is no universally correct pair.** The right values depend on your document and the questions asked of it. Tuning them is the honest, unglamorous work that separates a RAG demo from a RAG system, and it is what the next labs attack with smarter chunking, hybrid search, and reranking.

---

### Step 7 — Ask a Question the Document Cannot Answer

Step 4 gave us three chunks that all looked fine. This step finds out what "not fine" actually looks like, by asking about something this document has no opinion on.

In [11]:
# A question with no possible answer in a wetland-restoration guide.
OFF_TOPIC_QUESTION = "Who won the 1998 FIFA World Cup?"

off_topic_matches = vector_store.similarity_search_with_score(OFF_TOPIC_QUESTION, k=1)
off_topic_score = off_topic_matches[0][1]

print(f"off-topic best match: {off_topic_score:.3f}")
print(f"on-topic best match:  {matches[0][1]:.3f}")

off-topic best match: 0.021
on-topic best match:  0.771


- **The retriever still returned a chunk.** It always does — that is the behaviour from Step 4, now aimed at a question the document cannot possibly answer.
- **Look at the score: `0.021`, against `0.771` for the real question.** Nothing in Step 4's output told you which chunks were useful, because all of them scored above 0.48. This is the number that gives you a floor.
- **This is how you build an "I don't know" gate in a real system.** Pick a threshold — somewhere well below the scores you saw in Step 4, and well above `0.021` — and when the best match falls under it, stop retrieving and make the model say it does not know. Without that gate, a system that has never seen a document will still confidently retrieve its least-unrelated paragraph and answer from it.
- **You can run this for free.** No LLM call is involved, which is the point: measuring retrieval quality should not cost you anything.

---

## What We Learnt

- **RAG is an open-book exam** — find the right page first, then answer from that page only. Retrieval + supplied context + generation, in that order.
- **The loader is four lines of Python** — a `.txt` file needs no library, so `pathlib` does it. Every loader's only real job is handing you a `Document` with `page_content` and `metadata`; once you have one, the format of the original file stops mattering.
- **`\f` is a form feed, not a literal backslash-f** — it is how plain text marks a page break, and writing `"\\f"` splits on the wrong thing and quietly returns a single page.
- **The two models have different jobs** — the local embedding model turns text into comparable numbers; the remote LLM writes the answer. Steps 1–4, 6 and 7 need no API key at all.
- **The six stages are independent** — load, chunk, embed, index, retrieve, generate. Each is a separate step, and each is a place a later lab can improve without touching the others.
- **Embeddings turn meaning into geometry** — similar ideas land in similar directions, which is how search finds paraphrases instead of matching exact words.
- **The same embedding model must be used for the document *and* the question** — vectors from two different models are not comparable, and mixing them fails silently.
- **Unit vectors make cosine similarity a dot product** — which is why `MAX_INNER_PRODUCT` returns a score you can read directly as "how similar are these".
- **Read the metric, not its name** — `MAX_INNER_PRODUCT` gives cosine where higher is better, while `COSINE` gives squared L2 distance where lower is better. Same index library, opposite direction, and the wrong one silently ruins every score you print.
- **Similarity is not relevance** — all three chunks in Step 4 scored between 0.485 and 0.771 and only one answered the question. A cosine tells you two texts sound alike; reading them tells you whether they help.
- **A retriever always returns `k` results** — relevance is decided by the score, not by the result count, so even an unanswerable question comes back with a chunk attached.
- **Measure your floor before you trust your ceiling** — asking a question the document cannot answer scored `0.021`. That number is what lets a real system say "I don't know" instead of retrieving its least-unrelated paragraph and answering confidently.
- **Chunk size and top-k are trade-offs, not settings** — small chunks sharpen matching but lose context; more chunks raise recall at the cost of noise. There is no correct pair, only a better one for your documents.
- **Generation quality is capped by retrieval quality** — if the right chunk was not retrieved, no prompt can recover the answer. When an answer looks wrong, print the context before you touch the prompt.
- **Printing your sources is what makes RAG trustworthy** — the answer ships with the exact chunks and scores behind it, so grounding can be checked rather than assumed.